# Assignment 4: raw file to model-ready data
**Dataset:** Bengaluru House Price Data (Kaggle). Source link: *paste the Kaggle URL here*

Target column: `price` (in lakhs of rupees). No model is trained. The notebook stops at a clean matrix.

In [ ]:
import re
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler

pd.set_option("display.max_columns", 30)
FILE = "Bengaluru_House_Data.csv"   # upload to Colab first

## Q1. Data dictionary

In [ ]:
data_dictionary = pd.DataFrame([
    ["area_type",    "Kind of area the size is measured on",         "category", "n/a",        "4 labels: Super built-up Area, Built-up Area, Plot Area, Carpet Area"],
    ["availability", "Ready to move, or the date it will be ready",  "text",     "n/a",        "'Ready To Move' or a date such as '19-Dec'"],
    ["location",     "Neighbourhood in Bengaluru",                   "category", "n/a",        "about 1,300 distinct places"],
    ["size",         "Home size as text, e.g. '2 BHK', '4 Bedroom'", "text",     "bedrooms",   "1 to about 20"],
    ["society",      "Name of the housing society",                  "category", "n/a",        "mostly missing"],
    ["total_sqft",   "Total area; sometimes a range or other unit",  "text",     "square feet","about 250 to 50,000 once converted"],
    ["bath",         "Number of bathrooms",                          "numeric",  "count",      "1 to about 15"],
    ["balcony",      "Number of balconies",                          "numeric",  "count",      "0 to 3"],
    ["price",        "Price of the property (the target)",           "numeric",  "lakh rupees","about 8 to 3,600"],
], columns=["column", "meaning", "type", "unit", "expected range"])
data_dictionary

## Q2. Raw quality report

In [ ]:
raw = pd.read_csv(FILE)
print("shape:", raw.shape)

def to_sqft(x):
    """Convert the messy total_sqft text to square feet. Returns NaN if it cannot be read."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    try:
        return float(s)
    except ValueError:
        pass
    m = re.fullmatch(r"([\d.]+)\s*-\s*([\d.]+)", s)           # a range like 1133 - 1384
    if m:
        return (float(m.group(1)) + float(m.group(2))) / 2
    m = re.fullmatch(r"([\d.]+)\s*([A-Za-z. ]+)", s)            # a number with a unit
    if m:
        factors = {"sq. meter": 10.7639, "sq. yards": 9.0, "perch": 272.25,
                   "acres": 43560.0, "guntha": 1089.0, "grounds": 2400.0, "cents": 435.6}
        unit = m.group(2).strip().lower()
        if unit in factors:
            return float(m.group(1)) * factors[unit]
    return np.nan

sqft_num = raw["total_sqft"].apply(to_sqft)
bhk_num = pd.to_numeric(raw["size"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")

report = {
    "missing values (total cells)":             int(raw.isna().sum().sum()),
    "  missing in society":                     int(raw["society"].isna().sum()),
    "  missing in balcony":                     int(raw["balcony"].isna().sum()),
    "  missing in bath":                        int(raw["bath"].isna().sum()),
    "  missing in size":                        int(raw["size"].isna().sum()),
    "  missing in location":                    int(raw["location"].isna().sum()),
    "exact duplicate rows":                     int(raw.duplicated().sum()),
    "total_sqft not a plain number":            int(pd.to_numeric(raw["total_sqft"], errors="coerce").isna().sum()),
    "  of those, ranges like '1133 - 1384'":    int(raw["total_sqft"].astype(str).str.contains(r"\d\s*-\s*\d").sum()),
    "  of those, other units (Sq. Meter, Perch...)": int((pd.to_numeric(raw["total_sqft"], errors="coerce").isna()
                                                       & ~raw["total_sqft"].astype(str).str.contains(r"\d\s*-\s*\d")).sum()),
    "size written as 'Bedroom' instead of 'BHK'": int(raw["size"].astype(str).str.contains("Bedroom").sum()),
    "impossible: bath <= 0":                    int((raw["bath"] <= 0).sum()),
    "impossible: price <= 0":                   int((raw["price"] <= 0).sum()),
    "implausible: under 300 sqft per bedroom":  int((sqft_num / bhk_num < 300).sum()),
    "implausible: bathrooms > bedrooms + 2":    int((raw["bath"] > bhk_num + 2).sum()),
    "inconsistent labels in availability":      int(raw["availability"].nunique()),
    "location labels that differ only by spaces/case": int(raw["location"].nunique()
                                                         - raw["location"].str.strip().str.lower().nunique()),
}
pd.Series(report, name="count").to_frame()

## Q3. New features (one line of reasoning each)

| feature | how it is built | reasoning |
|---|---|---|
| `sqft_per_bhk` | `total_sqft / bhk` | Two 3 BHK flats can be very different. Room size says more about price than room count. |
| `bath_per_bhk` | `bath / bhk` | A rough measure of how luxurious the layout is. |
| `bath_extra` | `bath - bhk` | Flats with more bathrooms than bedrooms are usually premium, and the value also exposes data entry errors. |
| `is_ready_to_move` | `availability == 'Ready To Move'` | The raw column has dozens of date strings. Ready or not is the part that matters. |
| `has_society` | `society.notna()` | 41% of society values are missing, so the name is useless, but whether one exists may signal a managed, gated property. |

I deliberately did **not** build `price_per_sqft`, because it is made from the target and would leak the answer into X.

## Q4. The four functions (one cell each)

In [ ]:
def load_data():
    """Read the raw CSV and return it untouched."""
    return pd.read_csv(FILE)

In [ ]:
def clean(df):
    """Row-level cleaning that needs nothing learned from the data: fix text, units, impossible values."""
    df = df.copy()
    df = df.drop_duplicates()

    # strip whitespace and fix the case so labels match
    for col in ["area_type", "availability", "location", "society"]:
        df[col] = df[col].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    df["location"] = df["location"].str.title()

    # size: '2 BHK' and '4 Bedroom' both become the number of bedrooms
    df["bhk"] = pd.to_numeric(df["size"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
    df = df.drop(columns="size")

    # total_sqft: ranges and other units become one number in square feet
    df["total_sqft"] = df["total_sqft"].apply(to_sqft)

    # impossible values: turn into NaN, or drop the row if the target or area is unusable
    df.loc[df["bath"] <= 0, "bath"] = np.nan
    df = df[df["price"].notna() & (df["price"] > 0)]
    df = df[df["total_sqft"].notna() & (df["total_sqft"] > 0) & df["bhk"].notna()]

    # implausible rows: less than 300 sqft per bedroom, or far more bathrooms than bedrooms
    df = df[df["total_sqft"] / df["bhk"] >= 300]
    df = df[~(df["bath"] > df["bhk"] + 2)]
    return df.reset_index(drop=True)

In [ ]:
def add_features(df):
    """Create the new columns from Q3 and drop columns we no longer need."""
    df = df.copy()
    df["sqft_per_bhk"]     = df["total_sqft"] / df["bhk"]
    df["bath_per_bhk"]     = df["bath"] / df["bhk"]
    df["bath_extra"]       = df["bath"] - df["bhk"]
    df["is_ready_to_move"] = (df["availability"] == "Ready To Move").astype(int)
    df["has_society"]      = df["society"].notna().astype(int)
    return df.drop(columns=["availability", "society"])

In [ ]:
NUM_COLS = ["total_sqft", "bath", "balcony", "bhk", "sqft_per_bhk", "bath_per_bhk", "bath_extra"]
CAT_COLS = ["area_type", "location"]
BIN_COLS = ["is_ready_to_move", "has_society"]

def build_pipeline():
    """Imputing, encoding and scaling in one ColumnTransformer. Nothing is fitted here."""
    num = Pipeline([("impute", SimpleImputer(strategy="median")),
                    ("scale",  RobustScaler())])
    cat = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                             min_frequency=10, sparse_output=False))])
    return ColumnTransformer([("num", num, NUM_COLS),
                              ("cat", cat, CAT_COLS),
                              ("bin", "passthrough", BIN_COLS)])

## Q5. Raw file to `X_train`, `X_test`, `y_train`, `y_test` in one cell

In [ ]:
raw = load_data()
df = add_features(clean(raw))

X = df.drop(columns="price")
y = df["price"]
X_tr_raw, X_te_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = build_pipeline()
X_train = pipe.fit_transform(X_tr_raw)   # fitted on train only
X_test  = pipe.transform(X_te_raw)

for name, obj in [("X_train", X_train), ("X_test", X_test), ("y_train", y_train), ("y_test", y_test)]:
    print(name, obj.shape)

## Q6. Proof that the output is clean

In [ ]:
assert not np.isnan(X_train).any(), "NaN left in X_train"
assert not np.isnan(X_test).any(),  "NaN left in X_test"
assert not y_train.isna().any() and not y_test.isna().any(), "NaN in target"
assert X_train.shape[1] == X_test.shape[1], "train and test have different column counts"
assert (X_train.std(axis=0) > 0).all(), "a constant column exists"
print("All checks passed")

## Q7. Before and after

In [ ]:
n_final_rows = X_train.shape[0] + X_test.shape[0]
before_after = pd.DataFrame({
    "raw stage":   [raw.shape[0], raw.shape[1], int(raw.isna().sum().sum()), int(raw.duplicated().sum())],
    "final stage": [n_final_rows, X_train.shape[1] + 1, int(np.isnan(X_train).sum() + np.isnan(X_test).sum()), 0],
}, index=["row count", "column count (final includes the target)", "missing value count", "duplicate count"])
before_after

## Q8. Save the cleaned data and the fitted pipeline

In [ ]:
df.to_csv("bengaluru_clean.csv", index=False)
joblib.dump(pipe, "pipeline.joblib")

from google.colab import files
files.download("bengaluru_clean.csv")
files.download("pipeline.joblib")

## Q9. Decisions and the number behind each
*Replace the bracketed numbers with the ones printed in your own report above.*

- Dropped `society`, [about 41]% missing. Kept a `has_society` flag instead.
- Dropped [N] exact duplicate rows.
- Converted `total_sqft` to numbers, [N] values were ranges (took the midpoint) and [N] were in other units (converted to sqft).
- Turned `size` into `bhk`, [N] rows said "Bedroom" instead of "BHK".
- Removed [N] rows with under 300 sqft per bedroom, a plausibility rule.
- Removed [N] rows with more than 2 extra bathrooms than bedrooms.
- Filled `balcony` ([about 4.6]% missing) and `bath` ([about 0.5]% missing) with the training median.
- Grouped locations with under 10 training rows into one "infrequent" group, because [N] of [~1,300] locations have very few rows.
- Used RobustScaler for numeric columns, because `total_sqft` and `price` have long right tails.
- Split 80/20 with random_state=42 before fitting anything.

## Q10. What is still wrong
*Write this in your own words. Points that are true for this dataset:* `price` has no date or year, so prices from different periods are mixed together. `area_type` changes what `total_sqft` means (carpet area and super built-up area are not comparable). `location` is very high cardinality and many places have only a handful of listings, so the model will be weak there. There is no information on floor, age of building or amenities, which strongly affect price. A few extreme prices could be typing errors but cannot be confirmed without the source.